# 技能1 · Day 1 上机：营销文本表示学习

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **sentence-transformers** 将营销评论编码为 384 维 embedding
2. 用 **scikit-learn** 做 t-SNE/PCA 降维可视化和 KMeans 聚类，理解表示空间的几何结构
3. 用 **torch** 实现 Autoencoder 压缩表示，理解"压缩-重建"瓶颈
4. 用 **DSR 六步框架**定义"企业表示工程"研究问题，把工程实践转化为学术贡献

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：sentence-transformers（UKPLab/sentence-transformers，18.9k★）+ scikit-learn + torch。
营销映射：20条产品评论（护肤/电子/健身 × 正面/负面），用 embedding 编码后做降维/压缩/聚类/分类。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ sentence-transformers 首次运行会自动下载 all-MiniLM-L6-v2 模型（约 90MB），需网络。
> 模型缓存到 ~/.cache/huggingface/，后续运行无需网络。
> 如需更好的中文支持，可将模型名替换为 `paraphrase-multilingual-MiniLM-L12-v2`（同为 384 维）。

In [ ]:
# !pip install sentence-transformers scikit-learn torch -q

## 1. 数据集背景与营销映射

**处理对象**：20 条真实营销场景的产品评论（护肤/电子/健身三类 × 正面/负面两种情感）。

| 类别 | 正面 | 负面 | 示例 |
|------|------|------|------|
| 护肤 | 4条 | 4条 | "这款烟酰胺精华液真的太好用了，用了两周肤色明显提亮..." |
| 电子 | 3条 | 3条 | "跑步手表功能很全面，GPS轨迹精准，续航14天不用充..." |
| 健身 | 3条 | 3条 | "瑜伽垫材质很好，防滑效果一流，做下犬式再也不滑了..." |

每条评论包含：
- `review`：评论文本（中文，50-100字）
- `category`：产品类别（skincare / electronics / fitness）
- `sentiment`：情感标签（positive / negative）

**营销映射**：在真实项目中，这些数据来自电商平台的用户评价系统。表示工程的目标是把文本评论转化为可计算的向量，用于客户分群、情感分析、推荐系统。

**理论连接**：从 `f(x)=wᵀφ(x)`（手工特征）到 `f(x)=wᵀφ_θ(x)`（端到端学习）的范式转移--sentence-transformers 的 `φ_θ` 是预训练的，直接将文本映射为语义向量。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np

# sentence-transformers: 将文本编码为 embedding
from sentence_transformers import SentenceTransformer

# scikit-learn: 降维、聚类、评估
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# torch: 自编码器
import torch
import torch.nn as nn

print("依赖导入完成")

## TODO 1：用 sentence-transformers 编码营销评论

In [ ]:
# TODO 1：用 sentence-transformers 编码营销评论为 embedding
# 提示：用 SentenceTransformer('all-MiniLM-L6-v2') 加载模型
#   model.encode(reviews, show_progress_bar=True) 返回 numpy 数组
#   查看维度：embeddings.shape 应为 (20, 384)
# 要求：加载模型，编码20条评论，打印 embedding 维度

# ===== 你的代码 =====
model = None        # TODO: 加载 SentenceTransformer 模型
embeddings = None   # TODO: 编码 reviews 为 embedding
# ====================

print(f"Embedding 维度: {embeddings.shape}")
print(f"单条评论向量示例（前10维）: {embeddings[0][:10].round(4)}")

## 2. 表示学习理论基础回顾

### 范式转移：从手工特征到端到端学习

```
传统范式：f(x) = wᵀ φ(x)       -- φ 人工设计、固定
端到端范式：f(x) = wᵀ φ_θ(x)   -- φ_θ 数据驱动学习（θ 是可训练参数）
```

在营销场景中：传统方法需手动设计"浏览次数""购物车金额"等特征；端到端学习从原始行为文本中自动学习"比较后收藏但未购买 = 等待降价意图"这类序列模式。

### CMU 10741 三个核心概念

1. **不加约束的表示学习没有意义**：不限制维度，模型会退化为 lookup table（记忆而非学习）。约束（如 384 维）迫使模型发现潜在结构。
2. **Neural Collapse**（Papyan et al., 2020）：分类网络训练后期，最后一层特征呈现类别内方差趋零、类别间距离最大化的几何结构。好的表示让"相似的聚在一起，不同的分开"。
3. **不可辨识性**：不同随机种子训练的 embedding 数值不同但语义等价。不能解释单个维度，应关注几何关系（余弦相似度）。

### 非线性降维：为什么要用 t-SNE

t-SNE 用 t 分布（而非高斯分布）度量低维空间相似度，t 分布有更重的尾巴，解决了"中等距离的点降维后全挤一起"的拥挤问题。营销应用：把客户 embedding 投影到 2D，观察群体结构。

## TODO 2-3：降维可视化 + 自编码器压缩

In [ ]:
# TODO 2：用 t-SNE 和 PCA 降维可视化，观察正负面评论聚类结构
# 提示：PCA(n_components=2) -> fit_transform(embeddings)
#       TSNE(n_components=2, perplexity=5, random_state=42) -> fit_transform(embeddings)
#       perplexity 必须 < 样本数（20），用 5 较安全
# 要求：用两种方法降维，打印每条评论的2D坐标和情感标签

# ===== 你的代码 =====
embeddings_pca = None   # TODO: PCA 降维到 2D
embeddings_tsne = None  # TODO: t-SNE 降维到 2D
# ====================

print("PCA 降维结果：")
for i in range(len(reviews)):
    print(f"  评论{i+1:2d} [{sentiments[i]:8s}] PCA=({embeddings_pca[i,0]:.2f}, {embeddings_pca[i,1]:.2f})")

print("\nt-SNE 降维结果：")
for i in range(len(reviews)):
    print(f"  评论{i+1:2d} [{sentiments[i]:8s}] tSNE=({embeddings_tsne[i,0]:.2f}, {embeddings_tsne[i,1]:.2f})")

In [ ]:
# TODO 3：用 torch 实现 Autoencoder 压缩 embedding（384 -> 64），理解重构损失
# 提示：定义 Autoencoder(nn.Module)，包含 encoder (384->128->64) 和 decoder (64->128->384)
#       用 MSELoss 计算重构误差，Adam 优化器训练 200 轮
#       训练后获取压缩表示 z（64维）
# 要求：定义 Autoencoder 类，训练，打印重构损失和压缩表示维度

# ===== 你的代码 =====
class Autoencoder(nn.Module):
    def __init__(self, input_dim=384, latent_dim=64):
        super().__init__()
        # TODO: 定义 encoder 和 decoder
        pass

    def forward(self, x):
        # TODO: 返回 (重构输出, 压缩表示 z)
        pass

autoencoder = None  # TODO: 初始化模型
# 训练循环
# TODO: 训练 200 轮，每 50 轮打印 loss
# 获取压缩表示
compressed = None  # TODO: 获取压缩后的表示
# ====================

print(f"压缩表示维度: {compressed.shape}")
print(f"最终重构损失: {loss.item():.6f}")

## 3. 聚类与表示质量

### KMeans + Silhouette 评估

KMeans 在 embedding 空间中分群，Silhouette Score 衡量"类内紧凑、类间分离"的程度：

```
s(i) = (b(i) - a(i)) / max(a(i), b(i))
```

其中 `a(i)` 是样本到同簇其他点的平均距离，`b(i)` 是样本到最近其他簇的平均距离。s 越接近 1 表示聚类越好。

### 表示质量评估的两种方式

1. **无监督（内部指标）**：Silhouette Score --不需要标签，直接看向量空间的聚类结构
2. **有监督（下游任务）**：用 embedding 做情感分类，准确率高 = 表示质量好

这两种方式互补：无监督指标看"表示空间的几何结构"，有监督指标看"表示对下游任务的有效性"。

## TODO 4：KMeans 聚类发现评论分群

In [ ]:
# TODO 4：用 KMeans 聚类发现评论分群，silhouette 评估最优 K
# 提示：尝试 K=2,3,4,5，用 silhouette_score(embeddings, labels) 评估
#       KMeans(n_clusters=k, random_state=42, n_init=10)
# 要求：遍历不同 K 值，打印 Silhouette Score，找出最优 K，做最终聚类

# ===== 你的代码 =====
best_k = None      # TODO: 找出最优 K
best_score = None  # TODO: 最优 Silhouette Score
cluster_labels = None  # TODO: 最终聚类标签
# ====================

print(f"\n最优 K 值: {best_k}, Silhouette Score: {best_score:.4f}")
print("\n聚类结果 vs 情感标签：")
for i in range(len(reviews)):
    print(f"  评论{i+1:2d} [真实:{sentiments[i]:8s}] -> 聚类{cluster_labels[i]}")

## 4. DSR 六步框架：从工程实践到学术贡献

设计科学研究（Design Science Research）是信息系统领域的核心研究范式（Hevner 2004 / Peffers 2007），通过设计和评估 artifact 产生新知识。

**六步流程**：
1. 问题识别与动机 -- 从"标签"到"向量"的 gap
2. 定义解决方案目标 -- 统一表示框架
3. 设计与开发 -- embedding 系统 + Two-Tower + 对比学习
4. 演示 -- 在真实营销场景中验证
5. 评估 -- Recall@K / Silhouette / 跨域匹配准确率
6. 传播 -- IMRaD 论文投稿

**关键思考**：DSR 让工程实践有学术贡献框架--不是"做了系统"，而是"设计了新框架并验证有效性"。这正是博士级研究和硕士级项目的本质区别。

## TODO 5：用 DSR 六步框架定义研究问题

In [ ]:
# TODO 5：用 DSR 六步框架定义"企业表示工程"研究问题
# 提示：参考 Hevner 2004 / Peffers 2007 的六步流程
#   定义一个字典，包含六个步骤的具体内容
# 要求：为每一步填写"企业表示工程"的具体内容，打印框架

# ===== 你的代码 =====
dsr_framework = None  # TODO: 定义 DSR 六步框架字典
# ====================

for step, content in dsr_framework.items():
    print(f"{step}:")
    print(f"  {content}\n")

## 5. 表示质量评估

好的表示应该让下游任务表现好。我们用两种方式评估：

1. **下游分类准确率**：用 embedding 做情感分类（positive/negative），用 5 折交叉验证计算准确率
2. **Silhouette 对比**：比较原始 384 维表示和 Autoencoder 压缩后的 64 维表示的 Silhouette Score

**期望结果**：
- 原始 384 维表示的分类准确率应高于随机（50%）
- 压缩到 64 维后，准确率可能略降但不应大幅下降--说明 Autoencoder 保留了核心信息
- Silhouette Score 的变化反映压缩对表示空间几何结构的影响

## TODO 6：评估表示质量（下游分类 + Silhouette 对比）

In [ ]:
# TODO 6：评估表示质量（下游分类准确率 + Silhouette 对比）
# 提示：1) 用 LogisticRegression + cross_val_score(cv=5) 做情感分类
#       2) 对比原始 384 维 vs 压缩 64 维的分类准确率
#       3) 对比两者的 Silhouette Score（用情感标签作为真实标签）
# 要求：打印两种表示的分类准确率和 Silhouette Score

y = np.array([1 if s == 'positive' else 0 for s in sentiments])

# ===== 你的代码 =====
# 原始 384 维表示
clf = None           # TODO: LogisticRegression
scores_orig = None   # TODO: cross_val_score on 384-dim embeddings

# 压缩 64 维表示
compressed_np = None  # TODO: 将 torch tensor 转 numpy
scores_comp = None    # TODO: cross_val_score on 64-dim compressed

# Silhouette 对比
sil_orig = None   # TODO: silhouette_score on 384-dim
sil_comp = None   # TODO: silhouette_score on 64-dim
# ====================

print("=" * 55)
print(f"原始 384 维表示：分类准确率 = {scores_orig.mean():.4f} +/- {scores_orig.std():.4f}")
print(f"压缩  64 维表示：分类准确率 = {scores_comp.mean():.4f} +/- {scores_comp.std():.4f}")
print(f"原始 384 维 Silhouette = {sil_orig:.4f}")
print(f"压缩  64 维 Silhouette = {sil_comp:.4f}")
print("=" * 55)

## 6. 反思与前沿

### 反思问题
1. 你的营销评论 embedding 在 t-SNE 降维后呈现什么聚类结构？正负面评论是否清晰分离？
2. Autoencoder 压缩到 64 维后，重构损失是多少？信息损失多大？
3. KMeans 聚类的最优 K 是多少？聚类结果与产品类别/情感标签的对应关系如何？
4. 原始 384 维和压缩 64 维表示在下游分类准确率上差异多大？说明什么？

### 2026 前沿：Representation Engineering（RepE）
Representation Engineering（Zou et al., 2023, arXiv 2310.01405）是 MIT/Center for AI Safety 提出的 AI 透明性方法，通过操控神经网络内部的高层表示来监测和干预模型行为：
- **表示监测**：读取模型内部表示，判断是否在"想"虚构信息--比只看输出更早发现幻觉
- **表示操控**：调整内部表示方向，引导模型行为（增强品牌调性、抑制夸大宣传）
- **与本 Day 的连接**：RepE 的理论基础正是表示学习--只有理解了 embedding 空间的几何结构，才能理解如何读取和操纵表示

> ⚠️ RepE 对应因果阶梯 L1（关联分析），不能替代 L2（A/B 测试）。定位为"开发期透明性工具"。

参考 [arXiv 2310.01405](https://arxiv.org/abs/2310.01405)（Representation Engineering）+ [Neural Collapse](https://arxiv.org/abs/2008.08186)（Papyan et al., 2020）。